# 结构化输出

目前LangChain 1.x 支持多种Schema与结构化输出方式：
- Pydantic（字段校验、描述、嵌套结构，功能最丰富）
- TypedDict（轻量类型约束）
- JSON Schema（与前后端/跨语言接口最通用）
- dataclass
模型对象可以调用 with_structured_output() 绑定输出模式（schema）。
只有 Pydantic 返回的是Schema类实例，其余三种方式返回的都是 字典 ；也只有 Pydantic 在类型不匹配时会抛出异常。

## Pydantic
它通过在运行时强制执行类型提示，确保数据的正确性和一致性，是 生产场景首选 。

需要满足的几个要素：
- 所有结构化输出的数据模型都必须继承 BaseModel
- 使用 类型提示 。Pydantic 支持丰富的字段类型：str 、int、float、List[xxx]、Optional[xxx]等
- 使用 Field() 添加字段默认值和描述，帮助 LLM 理解字段含义

In [3]:
from dotenv import  load_dotenv
from pydantic import BaseModel,Field
from typing import override
from langchain.chat_models import  init_chat_model

load_dotenv(override=True)

model=init_chat_model(model_provider="deepseek",
                      model="deepseek-flash",
                      reasoning_effort="none")

# 定义pydantic模型
class Person(BaseModel):
    name:str=Field(description="姓名")
    age:int=Field(description="年龄")
    occupation:str=Field(description="职业")

#使用 with_structured_output 即可引导模型进行结构化输出：

structured_llm=model.with_structured_output(Person)
result=structured_llm.invoke("张三是一名 30 岁的软件工程师")

print(result)
print(type(result))

name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [4]:
from pydantic import BaseModel, Field, SecretStr
class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")
model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=9.3
<class '__main__.MovieModel'>


In [5]:
from typing import  Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import  init_chat_model

load_dotenv(override=True)

model=init_chat_model(model_provider="deepseek",
                      model="deepseek-flash",
                      reasoning_effort="none")

class config(BaseModel):
    timeout:Optional[int]=Field(30,description="超时时间")
    retry:bool=Field(False,description="是否支持重试")
    max_attempts:int=Field(6,description="最大重试次数")

structured_llm=model.with_structured_output(config)
structured_llm.invoke("配置要求：支持重试,最多重试5次")


config(timeout=30, retry=True, max_attempts=5)

In [ ]:
# 枚举类型

from enum import  Enum
class Priority(str,Enum):
    low="低"
    medium="中"
    high="高"

class Task(BaseModel):
    title:str
    priority:Priority

In [7]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
# 定义你的优先级枚举类
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

# 测试
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency.value}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直未发货' urgency=<Priority.HIGH: '高'>

提取结果：
 客户: 王小明
 电话: 138-1234-5678
 邮箱: None
 问题: 订单一直未发货
 紧急程度: 高


### 直接用typing中的Literal替代Enum类

In [9]:
from enum import Enum
from typing import Optional,Literal
from pydantic import BaseModel, Field
# 定义你的优先级枚举类
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低","中","高"] = Field(description="紧急程度")

# 测试
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直未发货' urgency='高'

提取结果：
 客户: 王小明
 电话: 138-1234-5678
 邮箱: 未提供
 问题: 订单一直未发货
 紧急程度: 高


In [3]:
# 列表提取
from typing import List
from dotenv import  load_dotenv
from pydantic import BaseModel
from langchain.chat_models import  init_chat_model

load_dotenv(override=True)

model=init_chat_model(model_provider="deepseek",
                      model="deepseek-flash",
                      reasoning_effort="none")


class Person(BaseModel):
    name:str
    age:int

class PersonList(BaseModel):
    people:List[Person]

structured_llm=model.with_structured_output(PersonList)
result=structured_llm.invoke("rr 18岁，小rr 8岁")

print(result)

people=[Person(name='rr', age=18), Person(name='小rr', age=8)]


In [7]:
from typing import List
from dotenv import  load_dotenv
from pydantic import BaseModel,Field
from langchain.chat_models import  init_chat_model

load_dotenv(override=True)

model=init_chat_model(model_provider="deepseek",
                      model="deepseek-flash",
                      reasoning_effort="none")


class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")
structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")
print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


In [8]:
# 嵌套结构
from typing import List
from dotenv import  load_dotenv
from pydantic import BaseModel
from langchain.chat_models import  init_chat_model

load_dotenv(override=True)

model=init_chat_model(model_provider="deepseek",
                      model="deepseek-flash",
                      reasoning_effort="none")

class Address(BaseModel):
    city:str
    district:str

class Company(BaseModel):
    name:str
    address:Address

structured_llm=model.with_structured_output(Company)
result = structured_llm.invoke("阿里巴巴在杭州滨江区")
print(result)

name='阿里巴巴' address=Address(city='杭州', district='滨江区')
